# Stage 0 Query Parser — Fine-tuning on Colab

Fine-tunes **Qwen2.5-1.5B-Instruct** with QLoRA to replace the OpenAI API call in Stage 0.

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [1]:
!pip install -q torch transformers peft bitsandbytes trl datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.5 MB/s eta 0:00:00


In [ ]:
import json
import os
import torch
from google.colab import files
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [2]:
os.makedirs('training', exist_ok=True)

print('Upload chat_pairs.jsonl (training data):')
uploaded = files.upload()
for name, data in uploaded.items():
    path = f'training/{name}'
    with open(path, 'wb') as f:
        f.write(data)
    print(f'  Saved: {path} ({len(data)} bytes)')

print('\nUpload chat_pairs_eval.jsonl (eval data):')
uploaded = files.upload()
for name, data in uploaded.items():
    path = f'training/{name}'
    with open(path, 'wb') as f:
        f.write(data)
    print(f'  Saved: {path} ({len(data)} bytes)')

Upload chat_pairs.jsonl (training data):


Saving chat_pairs.jsonl to chat_pairs.jsonl
  Saved: training/chat_pairs.jsonl (357055 bytes)

Upload chat_pairs_eval.jsonl (eval data):


Saving chat_pairs_eval.jsonl to chat_pairs_eval.jsonl
  Saved: training/chat_pairs_eval.jsonl (64510 bytes)


In [3]:
with open('training/chat_pairs.jsonl') as f:
    train_data = [json.loads(l) for l in f if l.strip()]

eval_path = 'training/chat_pairs_eval.jsonl'
if os.path.exists(eval_path):
    with open(eval_path) as f:
        eval_data = [json.loads(l) for l in f if l.strip()]
else:
    eval_data = []

print(f'Train examples: {len(train_data)}')
print(f'Eval examples:  {len(eval_data)}')
print(f'\nSample query: {train_data[0]["messages"][1]["content"]}')
print(f'Sample output: {train_data[0]["messages"][2]["content"][:200]}...')

Train examples: 137
Eval examples:  25

Sample query: Food and beverage manufacturers in France
Sample output: {"locations": ["france"], "geo_lat": null, "geo_lng": null, "geo_radius_km": null, "employee_count_min": null, "employee_count_max": null, "revenue_min": null, "revenue_max": null, "year_founded_min":...


In [4]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig( load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True )
model = AutoModelForCausalLM.from_pretrained( MODEL_NAME, quantization_config=bnb_config, device_map='auto', trust_remote_code=True )
model = prepare_model_for_kbit_training(model)

print(f'Model loaded: {MODEL_NAME}')

GPU: NVIDIA L4
VRAM: 23.7 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [5]:
lora_config = LoraConfig( r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'] )

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

Trainable: 18,464,768 / 907,081,216 (2.04%)


In [6]:
dataset = load_dataset('json', data_files='training/chat_pairs.jsonl', split='train')

eval_dataset = None
if os.path.exists('training/chat_pairs_eval.jsonl'):
    eval_dataset = load_dataset('json', data_files='training/chat_pairs_eval.jsonl', split='train')

def format_chat(example):
    text = tokenizer.apply_chat_template( example['messages'], tokenize=False, add_generation_prompt=False )
    return {'text': text[:2048]}

dataset = dataset.map(format_chat)
if eval_dataset:
    eval_dataset = eval_dataset.map(format_chat)

print(f'Dataset ready: {len(dataset)} train, {len(eval_dataset) if eval_dataset else 0} eval')

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/137 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Dataset ready: 137 train, 25 eval


In [7]:
OUTPUT_DIR = 'models/query-parser'
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = SFTConfig( output_dir=OUTPUT_DIR, num_train_epochs=5, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, learning_rate=2e-4, weight_decay=0.01, warmup_ratio=0.1, logging_steps=5, save_strategy='epoch',
    eval_strategy='epoch' if eval_dataset else 'no', bf16=True, dataset_text_field='text', report_to='none')
trainer = SFTTrainer( model=model, args=training_args, train_dataset=dataset, eval_dataset=eval_dataset, processing_class=tokenizer )

print('Starting training...')
trainer.train()
print('Training complete!')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/137 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Epoch,Training Loss,Validation Loss
1,1.906652,0.928009
2,0.588496,0.329127
3,0.309553,0.283900
4,0.255016,0.268081
5,0.251421,0.263641


Training complete!


In [8]:
adapter_path = os.path.join(OUTPUT_DIR, 'adapter')
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

config = { 'base_model': MODEL_NAME, 'adapter_path': 'adapter', 'lora_rank': 16, 'lora_alpha': 32, 'max_seq_length': 512, 'training_samples': len(dataset), 'epochs': 5 }
with open(os.path.join(OUTPUT_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

total_mb = sum(
    os.path.getsize(os.path.join(dp, fn))
    for dp, _, fns in os.walk(adapter_path)
    for fn in fns
) / (1024*1024)

print(f'Adapter saved: {adapter_path}/ ({total_mb:.0f} MB)')

Adapter saved: models/query-parser/adapter/ (46 MB)


In [12]:
model.eval()

test_queries = [
    'Logistics companies in Romania',
    'Public software companies with more than 1,000 employees',
    'Clean energy startups founded after 2018 in Scandinavia',
    'Companies that could supply packaging for a cosmetics brand',
    'Tech companies near Berlin with revenue over $10 million',
    'B2B SaaS companies providing HR solutions in Europe',
]

system_prompt = tokenizer.apply_chat_template(
    [{'role': 'system', 'content': dataset[0]['messages'][0]['content']}],
    tokenize=False, add_generation_prompt=False
)

for query in test_queries:
    messages = [
        {'role': 'system', 'content': dataset[0]['messages'][0]['content']},
        {'role': 'user', 'content': query},
    ]
    input_text = tokenizer.apply_chat_template( messages, tokenize=False, add_generation_prompt=True )
    inputs = tokenizer(input_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate( **inputs, max_new_tokens=1000, temperature=0.0, do_sample=False, pad_token_id=tokenizer.pad_token_id )

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()

    print(f'\n{"─"*60}')
    print(f'Query: {query}')
    print(f'{"─"*60}')
    try:
        parsed = json.loads(response)
        for k, v in parsed.items():
            if v is not None and v != [] and v != '':
                print(f'  {k}: {v}')
    except json.JSONDecodeError:
        print(f'  [NOT VALID JSON]')
        print(f'  {response[:300]}')


────────────────────────────────────────────────────────────
Query: Logistics companies in Romania
────────────────────────────────────────────────────────────
  locations: ['romania']
  naics_codes: ['4836', '4975']
  complexity: simple

────────────────────────────────────────────────────────────
Query: Public software companies with more than 1,000 employees
────────────────────────────────────────────────────────────
  [NOT VALID JSON]
  {"locations": null, "geo_lat": null, "geo_lng": null, "geo_radius_km": null, "employee_count_min": 1000, "employee_count_max": null, "revenue_min": null, "revenue_max": null, "year_founded_min": null, "year_founded_max": null, "is_public": true, "industry_keywords": ["software as a service", "cloud 

────────────────────────────────────────────────────────────
Query: Clean energy startups founded after 2018 in Scandinavia
────────────────────────────────────────────────────────────
  [NOT VALID JSON]
  {"locations": ["denmark", "finland", "iceland

In [10]:
!zip -r query-parser-adapter.zip models/query-parser/
files.download('query-parser-adapter.zip')
print('\nDownloaded! Unzip into your project\'s models/ directory.')
print('Test locally: python finetune.py --test --model-path models/query-parser')

  adding: models/query-parser/ (stored 0%)
  adding: models/query-parser/checkpoint-45/ (stored 0%)
  adding: models/query-parser/checkpoint-45/rng_state.pth (deflated 26%)
  adding: models/query-parser/checkpoint-45/tokenizer_config.json (deflated 59%)
  adding: models/query-parser/checkpoint-45/optimizer.pt (deflated 22%)
  adding: models/query-parser/checkpoint-45/scheduler.pt (deflated 62%)
  adding: models/query-parser/checkpoint-45/trainer_state.json (deflated 73%)
  adding: models/query-parser/checkpoint-45/chat_template.jinja (deflated 71%)
  adding: models/query-parser/checkpoint-45/tokenizer.json (deflated 81%)
  adding: models/query-parser/checkpoint-45/training_args.bin (deflated 53%)
  adding: models/query-parser/checkpoint-45/README.md (deflated 65%)
  adding: models/query-parser/checkpoint-45/adapter_config.json (deflated 58%)
  adding: models/query-parser/checkpoint-45/adapter_model.safetensors (deflated 22%)
  adding: models/query-parser/config.json (deflated 26%)
  ad

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloaded! Unzip into your project's models/ directory.
Test locally: python finetune.py --test --model-path models/query-parser
